# Sirroz bemorlari omon qolish prognozini aniqlash

Jigar sirrozi bilan og'rigan bemorlarning klinik ko'rsatkichlari asosida uchta natija ehtimolligini bashorat qilamiz.

* `C` - tadqiqot oxirida tirik (senzuralangan)
* `CL` - jigar transplantatsiyasi tufayli senzuralangan (tirik)
* `D` - vafot etgan

**Metrika:** multi-class log loss (kichikroq bo'lgani yaxshi).

## 1. Kutubxonalar

In [ ]:
#dataset bilan ishlash uchun
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

#vizualizatsiya uchun
import matplotlib.pyplot as plt

#ML
from scipy.optimize import minimize
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss
from xgboost import XGBClassifier

## 2. Ma'lumotlarni yuklash

In [ ]:
#datasets
train = pd.read_csv('https://raw.githubusercontent.com/botirTurgonboyev/Sirroz-bemorlari-omon-qolish-prognozini-aniqlash/refs/heads/main/train.csv')
test = pd.read_csv('https://raw.githubusercontent.com/botirTurgonboyev/Sirroz-bemorlari-omon-qolish-prognozini-aniqlash/refs/heads/main/test.csv')
submission = pd.read_csv('https://raw.githubusercontent.com/botirTurgonboyev/Sirroz-bemorlari-omon-qolish-prognozini-aniqlash/refs/heads/main/sample_submission.csv')

In [ ]:
train.head()

* train.csv - o'qitish uchun ma'lumotlar (15 000 qator, nishon: `Status`);
* test.csv - bashorat qilinadigan ma'lumotlar (10 000 qator);
* sample_submission.csv - topshirish formati: `id, Status_C, Status_CL, Status_D`.

In [ ]:
print(train.shape, test.shape)
train.info()

In [ ]:
train.describe()

## 3. Yo'qolgan qiymatlar

In [ ]:
miss = pd.DataFrame({'train': train.isnull().mean(), 'test': test.isnull().mean()}).round(3)
miss[(miss['train'] > 0) | (miss['test'] > 0)]

* `Drug`, `Ascites`, `Hepatomegaly`, `Spiders`, `Copper`, `Alk_Phos`, `SGOT`, `Cholesterol`, `Tryglicerides` taxminan 43-56% qatorda bo'sh, test'da ham xuddi shunday.
* Bu ustunlar odatda **birga** bo'sh bo'ladi (blok tuzilishi): quyida tekshiramiz.
* Bo'sh qatorlarni **tashlamaymiz**: test'ning ~44% qatorida ham shu ustunlar bo'sh va ularning hammasi uchun bashorat kerak. Bo'sh qiymatlarni modelning o'ziga topshiramiz.

In [ ]:
blk = ['Drug', 'Ascites', 'Hepatomegaly', 'Spiders', 'Alk_Phos',
       'SGOT', 'Copper', 'Cholesterol', 'Tryglicerides']
train[blk].isnull().sum(axis=1).value_counts().sort_index()

## 4. Xato qiymatlar

In [ ]:
age_yil = train['Age'] / 365.25
print('Age < 18 yil:', (age_yil < 18).sum(), '| Age > 90 yil:', (age_yil > 90).sum())
print('N_Days > 6000:', (train['N_Days'] > 6000).sum())
print('Age == N_Days (ko\'chirilgan):', (train['Age'] == train['N_Days']).sum())
print('Dublikat qatorlar:', train.drop(columns='id').duplicated().sum())

Xato qiymatlar kam (40 ga yaqin qator). Ularni `NaN` qilib tozalab ham sinab ko'rdik, CV natijasi yaxshilanmadi (0.3752 -> 0.3758), chunki daraxt modellari bunday chetlanishlarga chidamli. Shuning uchun **ma'lumotni o'zgartirmaymiz**.

## 5. Kategorik ustunlar

In [ ]:
#kategorik ustunlarni ajratib olamiz
n = 1
cat_col = []

for column in train.columns:
    if column in ['id', 'Status']:
        continue
    if not pd.api.types.is_numeric_dtype(train[column]):
        cat_col.append(column)
        print(f"{n} - {column}")
        n += 1

In [ ]:
for column in cat_col:
    print(train[column].value_counts(dropna=False))

In [ ]:
#kategorik ustunlarni o'zgartiramiz (NaN saqlanadi)
maps = {
    'Drug':         {'Placebo': 0, 'D-penicillamine': 1},
    'Sex':          {'F': 0, 'M': 1},
    'Ascites':      {'N': 0, 'Y': 1},
    'Hepatomegaly': {'N': 0, 'Y': 1},
    'Spiders':      {'N': 0, 'Y': 1},
    'Edema':        {'N': 0, 'S': 1, 'Y': 2},
}

for c, m in maps.items():
    before = train[c].isna().sum()
    train[c] = train[c].map(m)
    test[c] = test[c].map(m)
    print(c, 'NaN:', before, '->', train[c].isna().sum())

Bu katakni **bir marta** ishga tushiring. Ustunlar allaqachon raqam bo'lgani uchun ikkinchi marta ishga tushirsangiz hammasi `NaN` bo'lib qoladi.

## 6. Nishon (Status) taqsimoti

In [ ]:
from matplotlib.patches import Patch

ORDER  = ['C', 'CL', 'D']
COLORS = {'C': '#2a78d6', 'CL': '#1baf7a', 'D': '#eb6834'}
NAMES  = {'C': 'C - tirik', 'CL': 'CL - transplantatsiya', 'D': 'D - vafot'}
SURFACE, INK, INK2, MUTED, GRID = '#fcfcfb', '#0b0b0b', '#52514e', '#898781', '#e1e0d9'

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': INK2, 'text.color': INK,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'font.size': 10, 'figure.dpi': 110,
})

d = train[train['Status'].isin(ORDER)].copy()
d['Age_yil'] = d['Age'] / 365.25
d.loc[~d['Age_yil'].between(18, 90), 'Age_yil'] = np.nan
d.loc[d['N_Days'] > 6000, 'N_Days'] = np.nan
legend_handles = [Patch(facecolor=COLORS[s], label=NAMES[s]) for s in ORDER]
print(d.shape)

In [ ]:
counts = d['Status'].value_counts().reindex(ORDER)

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.bar(ORDER, counts.values, color=[COLORS[s] for s in ORDER], width=0.55)
for i, n in enumerate(counts.values):
    ax.text(i, n, f'{n:,}\n({n / counts.sum():.1%})', ha='center', va='bottom', color=INK)
ax.set_ylim(0, counts.max() * 1.22)
ax.set_title('Status taqsimoti')
ax.set_ylabel('Bemorlar soni')
ax.grid(axis='x', visible=False)
ax.legend(handles=legend_handles, frameon=False, loc='upper right')
plt.tight_layout()
plt.show()

* `C` ≈ 67%, `D` ≈ 30%, `CL` ≈ 2.5%, nishon nomutanosib.
* `Y` sinfi atigi 1 ta qator va `sample_submission.csv` da yo'q, uni olib tashlaymiz (modellashda).
* Vizualizatsiya kataklari kategorik ustunlar **kodlangandan keyin** ishga tushishi kerak (`d` nusxasi shu yerda yaratiladi).

## 7. Belgilar va Status bog'liqligi (vizualizatsiya)

In [ ]:
levels = {
    'Stage':        ([1, 2, 3, 4], ['1', '2', '3', '4']),
    'Edema':        ([0, 1, 2],    ['N', 'S', 'Y']),
    'Ascites':      ([0, 1],       ['N', 'Y']),
    'Hepatomegaly': ([0, 1],       ['N', 'Y']),
    'Spiders':      ([0, 1],       ['N', 'Y']),
}

fig, axes = plt.subplots(1, len(levels), figsize=(17, 4.6), sharey=True)
for ax, (col, (vals, labs)) in zip(axes, levels.items()):
    lab_map = dict(zip(vals, labs))
    g = d[col].map(lambda v: lab_map.get(v, "bo'sh") if pd.notna(v) else "bo'sh")
    ct = pd.crosstab(g, d['Status'], normalize='index').reindex(columns=ORDER) * 100
    n_per = g.value_counts()
    ct = ct.reindex([l for l in labs + ["bo'sh"] if l in ct.index])

    bottom = np.zeros(len(ct))
    for s in ORDER:
        ax.bar(ct.index, ct[s], bottom=bottom, color=COLORS[s],
               edgecolor=SURFACE, linewidth=1.5, width=0.7)
        bottom += ct[s].values
    for i, lab in enumerate(ct.index):          # har bir ustun ustiga D ulushi
        ax.text(i, 101, f"D {ct.loc[lab, 'D']:.0f}%", ha='center', va='bottom', color=INK, fontsize=9)
        ax.text(i, -17, f"n={n_per[lab]:,}", ha='center', va='top', color=MUTED, fontsize=8)
    ax.set_title(col)
    ax.set_ylim(0, 112)
    ax.grid(axis='x', visible=False)
axes[0].set_ylabel("Sinf ulushi, %")
fig.suptitle('Kategorik belgilar va Status', x=0.01, ha='left', fontsize=13, fontweight='bold')
fig.legend(handles=legend_handles, loc='upper right', ncol=3, frameon=False)
plt.tight_layout(rect=(0, 0.04, 1, 0.93))
plt.show()

* `D` ulushi `Stage` 1 da 7%, `Stage` 4 da 58%; `Edema = Y` da 99%; `Ascites = Y` da 97%.
* `bo'sh` toifadagi bemorlarda `D` ulushi (31%) o'rtachadan (30%) farq qilmaydi, ya'ni bo'shliqning o'zi kuchli belgi emas.

In [ ]:
num_cols = ['Bilirubin', 'Albumin', 'Copper', 'Prothrombin', 'Platelets', 'Age_yil', 'N_Days']
log_cols = {'Bilirubin', 'Copper'}
titles = {'Age_yil': 'Yosh (yil)', 'N_Days': 'N_Days (kun)'}

fig, axes = plt.subplots(2, 4, figsize=(17, 8))
axes = axes.ravel()
for ax, col in zip(axes, num_cols):
    data = [d.loc[d['Status'] == s, col].dropna() for s in ORDER]
    bp = ax.boxplot(data, positions=range(3), widths=0.55, patch_artist=True, showfliers=True,
                    medianprops=dict(color=INK, linewidth=1.8),
                    whiskerprops=dict(color=MUTED), capprops=dict(color=MUTED),
                    flierprops=dict(marker='o', markersize=2.5, markerfacecolor=MUTED,
                                    markeredgecolor='none', alpha=0.35))
    for patch, s in zip(bp['boxes'], ORDER):
        patch.set(facecolor=COLORS[s], edgecolor=SURFACE, linewidth=1.5)
    ax.set_xticks(range(3))
    ax.set_xticklabels([f"{s}\nn={len(x):,}" for s, x in zip(ORDER, data)])
    if col in log_cols:
        ax.set_yscale('log')
    ax.set_title(titles.get(col, col) + (' (log)' if col in log_cols else ''))
    ax.grid(axis='x', visible=False)
axes[-1].axis('off')
axes[-1].legend(handles=legend_handles, loc='center', frameon=False, fontsize=11)
fig.suptitle('Raqamli belgilar va Status (chiziq - median, quti - 25-75%)',
             x=0.01, ha='left', fontsize=13, fontweight='bold')
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

In [ ]:
Xc = d.drop(columns=['id', 'Status', 'Age'])
target = pd.get_dummies(d['Status']).astype(int)
corr = pd.DataFrame({s: Xc.corrwith(target[s], method='spearman') for s in ORDER})
order_idx = corr['D'].abs().sort_values().index        # eng kuchli tepada

fig, axes = plt.subplots(1, 2, figsize=(11, 7), sharey=True)
for ax, s in zip(axes, ['D', 'CL']):
    ax.barh(order_idx, corr.loc[order_idx, s], color=COLORS[s], height=0.6)
    for y, v in enumerate(corr.loc[order_idx, s]):
        ax.text(v + (0.008 if v >= 0 else -0.008), y, f'{v:+.2f}',
                ha='left' if v >= 0 else 'right', va='center', color=INK, fontsize=9)
    ax.axvline(0, color='#c3c2b7', linewidth=1)
    ax.set_xlim(-0.6, 0.6)
    ax.set_title(NAMES[s])
    ax.set_xlabel('Spearman korrelyatsiyasi')
    ax.grid(axis='y', visible=False)
fig.suptitle('Belgilarning Status bilan bog\'liqligi', x=0.01, ha='left', fontsize=13, fontweight='bold')
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

* `D` bilan eng kuchli bog'liq: `Bilirubin` (+0.50), `Prothrombin` (+0.45), `Hepatomegaly` (+0.43), `Copper` (+0.41), `Stage` (+0.39), `N_Days` (-0.35).
* `CL` signali kuchsiz: `Copper` (+0.12), `Age` (-0.12), `Bilirubin` (+0.10).
* `Drug` (-0.04) deyarli bog'liq emas, lekin uni tashlash CV natijasini yomonlashtirdi (~0.0015), shuning uchun **hamma ustunni qoldiramiz**.
* Korrelyatsiya bo'sh bo'lmagan qatorlarda hisoblangan.

## 8. Modellash uchun tayyorlash

In [ ]:
tr = train[train['Status'].isin(['C', 'CL', 'D'])].reset_index(drop=True)
features = [c for c in tr.columns if c not in ('id', 'Status')]

X = tr[features]
X_test = test[features]

le = LabelEncoder()
y = le.fit_transform(tr['Status'])
print('Sinflar tartibi:', le.classes_)
print('Train:', X.shape, '| Test:', X_test.shape)

cat_mask = [c in maps for c in features]

#har bir model uchun out-of-fold (OOF) va test bashoratlari shu yerga yig'iladi
oofs, tests = {}, {}
prior = np.bincount(y) / len(y)
print('Baseline (faqat sinf chastotasi) log loss:', round(log_loss(y, np.tile(prior, (len(y), 1))), 4))

* `id` belgilar orasiga **kirmaydi** (u faqat identifikator, kiritilsa natija yomonlashadi).
* `Status` ni `LabelEncoder` bilan raqamga o'tkazamiz: `C`=0, `CL`=1, `D`=2.
* Bo'sh qiymatlar `NaN` holida qoladi, daraxt modellari ularni o'zi qo'llaydi.

## 9. HistGradientBoosting (3 konfiguratsiya)

5-fold CV, 3 xil seed. Bu katak taxminan 3 daqiqa oladi.

In [ ]:
CONFIGS = {
    "hgb_small":   dict(learning_rate=0.03, max_leaf_nodes=8, min_samples_leaf=60,
                        l2_regularization=1.0, max_iter=2000),
    "hgb_medium":  dict(learning_rate=0.03, max_leaf_nodes=15, min_samples_leaf=40,
                        l2_regularization=2.0, max_iter=2000),
    "hgb_shallow": dict(learning_rate=0.02, max_depth=3, max_leaf_nodes=8,
                        min_samples_leaf=80, l2_regularization=3.0, max_iter=3000),
}
SEEDS = [42, 7, 2024]

def cv_hgb(params):
    oof = np.zeros((len(X), 3))
    tp = np.zeros((len(X_test), 3))
    for seed in SEEDS:
        skf = StratifiedKFold(5, shuffle=True, random_state=seed)
        for a, b in skf.split(X, y):
            model = HistGradientBoostingClassifier(
                categorical_features=cat_mask, early_stopping=True,
                validation_fraction=0.1, n_iter_no_change=60,
                random_state=seed, **params)
            model.fit(X.iloc[a], y[a])
            oof[b] += model.predict_proba(X.iloc[b]) / len(SEEDS)
            tp += model.predict_proba(X_test) / (len(SEEDS) * 5)
    return oof, tp

for name, params in CONFIGS.items():
    oofs[name], tests[name] = cv_hgb(params)
    print(f"{name:12s} CV log loss = {log_loss(y, oofs[name]):.4f}")

## 10. XGBoost

Nishon raqamli (`LabelEncoder`), daraxtlar soni `early_stopping_rounds` bilan tanlanadi (`GridSearchCV` bilan `n_estimators=1` chiqqan edi, chunki standart `learning_rate=0.3` juda tez ortiqcha moslashadi).

In [ ]:
# XGBOOST
xgb_params = dict(n_estimators=2000, learning_rate=0.03, max_depth=3, subsample=0.8,
                  colsample_bytree=0.8, min_child_weight=5, reg_lambda=3,
                  eval_metric='mlogloss', early_stopping_rounds=100,
                  random_state=42, n_jobs=-1)

oof_x = np.zeros((len(X), 3))
tp_x = np.zeros((len(X_test), 3))
skf = StratifiedKFold(5, shuffle=True, random_state=42)

for fold, (a, b) in enumerate(skf.split(X, y)):
    model = XGBClassifier(**xgb_params)
    model.fit(X.iloc[a], y[a], eval_set=[(X.iloc[b], y[b])], verbose=False)
    oof_x[b] = model.predict_proba(X.iloc[b])
    tp_x += model.predict_proba(X_test) / 5
    print(f'fold {fold}: daraxtlar={model.best_iteration}, log loss={log_loss(y[b], oof_x[b]):.4f}')

oofs['xgb'], tests['xgb'] = oof_x, tp_x
print('XGBoost CV log loss =', round(log_loss(y, oof_x), 4))

Erta to'xtatish validatsiya bo'lagida tanlangani uchun XGBoost bahosi juda oz optimistik bo'lishi mumkin.

## 11. ExtraTrees (xilma-xillik uchun)

In [ ]:
et_pipe = make_pipeline(
    SimpleImputer(strategy='median', add_indicator=True),
    ExtraTreesClassifier(n_estimators=400, min_samples_leaf=8, max_features=0.5,
                         n_jobs=-1, random_state=0))

oof_e = np.zeros((len(X), 3))
tp_e = np.zeros((len(X_test), 3))
for a, b in StratifiedKFold(5, shuffle=True, random_state=42).split(X, y):
    et_pipe.fit(X.iloc[a], y[a])
    oof_e[b] = et_pipe.predict_proba(X.iloc[b])
    tp_e += et_pipe.predict_proba(X_test) / 5

oofs['extratrees'], tests['extratrees'] = oof_e, tp_e
print('ExtraTrees CV log loss =', round(log_loss(y, oof_e), 4))

ExtraTrees `NaN` ni qabul qilmaydi, shuning uchun `SimpleImputer` median bilan to'ldiradi va `add_indicator=True` "bu qiymat bo'sh edi" belgilarini qo'shadi.

## 12. Ansambl

In [ ]:
names = list(oofs)
k = len(names)
mix = lambda w, P: sum(wi * P[n] for wi, n in zip(w, names))

res = minimize(lambda w: log_loss(y, mix(w, oofs)), np.ones(k) / k, method='SLSQP',
               bounds=[(0, 1)] * k, constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1})
w_opt = res.x

print(f"Eng yaxshi yakka model : {min(log_loss(y, oofs[n]) for n in names):.4f}")
print(f"Teng og'irlikli ansambl: {log_loss(y, mix(np.ones(k) / k, oofs)):.4f}")
print(f"Optimal ansambl        : {log_loss(y, mix(w_opt, oofs)):.4f}")
for n, w in zip(names, w_opt):
    print(f"   {n:12s} og'irlik = {w:.3f}")

oof_blend = mix(w_opt, oofs)
test_blend = mix(w_opt, tests)

Og'irliklar shu OOF bashoratlarida tanlangani uchun yakuniy baho haqiqiy test natijasidan biroz optimistik bo'lishi mumkin.

In [ ]:
#tekshiruv: bashorat qilingan o'rtacha ehtimolliklar haqiqiy sinf ulushlariga yaqinmi
fmt = lambda p: {c: round(float(v), 4) for c, v in zip(le.classes_, p)}
print('Haqiqiy ulush :', fmt(np.bincount(y) / len(y)))
print('OOF bashorat  :', fmt(oof_blend.mean(0)))
print('Test bashorat :', fmt(test_blend.mean(0)))

## 13. Submission

In [ ]:
cols = ['Status_C', 'Status_CL', 'Status_D']   # LabelEncoder tartibi: C, CL, D

final = np.clip(test_blend, 1e-4, 1 - 1e-4)
final = final / final.sum(axis=1, keepdims=True)

sub = pd.DataFrame(final, columns=cols)
sub.insert(0, 'id', test['id'].values)

assert list(sub.columns) == list(submission.columns)
assert sub['id'].tolist() == submission['id'].tolist()
assert np.allclose(sub[cols].sum(axis=1), 1) and not sub.isna().any().any()

sub.to_csv('submission.csv', index=False)
sub.head()

In [ ]:
#Colab'dan yuklab olish
try:
    from google.colab import files
    files.download('submission.csv')
except ImportError:
    pass

## Xulosa

* Bo'sh qatorlarni tashlamadik: test'ning ~44% qatorida shu ustunlar bo'sh va ularsiz o'qitilgan model bu qatorlarda sezilarli yomon ishlaydi.
* `id` ni belgi sifatida bermadik, `Y` sinfini olib tashladik.
* Tozalash va `Drug`/`Cholesterol`/`Tryglicerides` ni tashlash CV natijasini yaxshilamadi.
* Yakuniy bashorat: HistGradientBoosting, XGBoost va ExtraTrees ansambli, ehtimolliklar `[1e-4, 1-1e-4]` oralig'iga kesilgan.